In [ ]:
import pandas as pd
import numpy as np


# LOAD RAW EXCEL FILE
# Load the original .xls file.
# The dataset has a complex multi-row header, so it needs header=[2, 3] to capture the month structure.

file_path = r"Таблиця 1 23 Вихідні дані.xls"

df = pd.read_excel(file_path, header=[2, 3])

print("Raw shape:", df.shape)
print("\nFirst 10 raw columns:")
for col in df.columns[:10]:
    print(col)


# REBUILD COLUMN NAMES
# Combining: month from the header AND metric name from the first data row IN ORDER TO create usable column names such as: 01.01.2023__Початковий залишок

metric_row = df.iloc[0]

new_columns = []

for i, col in enumerate(df.columns):
    top, bottom = col

    if i == 0:
        new_columns.append("location")
    elif i == 1:
        new_columns.append("sku")
    elif i == 2:
        new_columns.append("date_introduced")
    elif i == 3:
        new_columns.append("last_release_date")
    else:
        month = bottom
        metric = metric_row.iloc[i]
        new_columns.append(f"{month}__{metric}")

df.columns = new_columns

print("\nNew columns created successfully.")
print(df.columns[:20].tolist())
print("Shape before dropping metric row:", df.shape)


# REMOVE HEADER-LIKE AND EMPTY ROWS
# The first row is not real data anymore because it was already used it to reconstruct the metric names. Then fully empty key rows and the total row "Разом" are removed.

# Drop the former metric row
df = df.iloc[1:].copy()

# Drop rows where both key identifiers are missing
df = df.dropna(subset=["location", "sku"], how="all").copy()

# Identify and remove the total row "Разом"
mask_total = (
    df["location"].astype(str).str.contains("Разом", case=False, na=False) |
    df["sku"].astype(str).str.contains("Разом", case=False, na=False)
)

print("\nRows with 'Разом':", mask_total.sum())
if mask_total.sum() > 0:
    print(df.loc[mask_total, ["location", "sku"]])

df = df.loc[~mask_total].copy()
df = df.reset_index(drop=True)

print("\nShape after initial cleaning:", df.shape)
print(df.head())


# CONVERT DATE COLUMNS


df["date_introduced"] = pd.to_datetime(
    df["date_introduced"],
    format="%d.%m.%Y",
    errors="coerce"
)

df["last_release_date"] = pd.to_datetime(
    df["last_release_date"],
    format="%d.%m.%Y",
    errors="coerce"
)

print("\nDate column types:")
print(df[["date_introduced", "last_release_date"]].dtypes)

print("\nMissing date_introduced:", df["date_introduced"].isna().sum())
print("Missing last_release_date:", df["last_release_date"].isna().sum())


# IDENTIFY COLUMN GROUPS
# Split columns into logical groups: IDs, date columns, ABC group columns, comments, balance check, numeric business metrics


id_columns = ["location", "sku"]
date_columns = ["date_introduced", "last_release_date"]

abc_columns = [col for col in df.columns if "АВС група" in col]
comment_columns = [col for col in df.columns if "Коментар" in col]
check_columns = [col for col in df.columns if "перевірка" in col]

excluded_columns = set(
    id_columns + date_columns + abc_columns + comment_columns + check_columns
)

numeric_columns = [col for col in df.columns if col not in excluded_columns]

print("\nColumn groups:")
print("ID columns:", len(id_columns))
print("Date columns:", len(date_columns))
print("ABC columns:", len(abc_columns))
print("Comment columns:", len(comment_columns))
print("Check columns:", len(check_columns))
print("Numeric columns:", len(numeric_columns))

print("\nFirst 10 numeric columns:")
print(numeric_columns[:10])


# CONVERT NUMERIC BUSINESS COLUMNS
# Convert all metric columns to numeric values, Non-numeric values are turned into NaN.


df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

print("\nNumeric column types:")
print(df[numeric_columns].dtypes.head(10))

missing_counts = df[numeric_columns].isna().sum().sort_values(ascending=False)
print("\nTop 15 columns by missing values:")
print(missing_counts.head(15))


# SAVE CLEANED WIDE VERSION


df.to_csv("inventory_clean_wide.csv", index=False, encoding="utf-8-sig")
print("\nFile saved: inventory_clean_wide.csv")


# PREPARE FOR WIDE-TO-LONG TRANSFORMATION


base_columns = ["location", "sku", "date_introduced", "last_release_date"]
value_columns = [col for col in df.columns if col not in base_columns]

print("\nBase columns:", base_columns)
print("Number of value columns:", len(value_columns))
print("First 10 value columns:")
print(value_columns[:10])


# MELT THE TABLE


df_melted = df.melt(
    id_vars=base_columns,
    value_vars=value_columns,
    var_name="month_metric",
    value_name="value"
)

print("\nShape of melted table:", df_melted.shape)
print(df_melted.head(10))


# 10. SPLIT month_metric INTO month AND metric
# Example: 01.01.2023__Початковий залишок becomes month = 01.01.2023

df_melted[["month", "metric"]] = df_melted["month_metric"].str.split(
    "__", n=1, expand=True
)

df_melted["month"] = pd.to_datetime(
    df_melted["month"],
    format="%d.%m.%Y",
    errors="coerce"
)

df_melted = df_melted.drop(columns=["month_metric"]).copy()

print("\nMelted table after splitting:")
print(df_melted.head(10))


# PIVOT BACK INTO FINAL ANALYTICAL LONG TABLE
# one location + one sku + one month


df_long = df_melted.pivot_table(
    index=["location", "sku", "date_introduced", "last_release_date", "month"],
    columns="metric",
    values="value",
    aggfunc="first"
).reset_index()

df_long.columns.name = None

print("\nShape of long table:", df_long.shape)
print(df_long.head())


# RENAME FINAL COLUMNS TO ENGLISH


rename_dict = {
    "АВС група": "ABC_group",
    "Видаток": "outflow",
    "Дод. замовлення": "extra_order",
    "Замовлення в цех": "production_order",
    "Заявка клієнтів": "customer_demand",
    "Коефіцієнт оборотності": "turnover_ratio",
    "Кінцевий залишок": "ending_stock",
    "Надходження": "inflow",
    "Початковий залишок": "opening_stock",
    "перевірка": "balance_check"
}

df_long = df_long.rename(columns=rename_dict)

print("\nFinal columns:")
print(df_long.columns.tolist())


# CONVERT FINAL NUMERIC COLUMNS

numeric_cols_long = [
    "outflow",
    "extra_order",
    "production_order",
    "customer_demand",
    "turnover_ratio",
    "ending_stock",
    "inflow",
    "opening_stock",
    "balance_check"
]

df_long[numeric_cols_long] = df_long[numeric_cols_long].apply(
    pd.to_numeric,
    errors="coerce"
)

print("\nFinal dtypes:")
print(df_long.dtypes)


# FINAL QUALITY CHECK
# Check for: final shape, duplicate key rows, missing values

print("\nShape:", df_long.shape)

duplicate_count = df_long.duplicated(
    subset=["location", "sku", "month"]
).sum()
print("Duplicates by location + sku + month:", duplicate_count)

print("\nMissing values:")
print(df_long.isna().sum())


# EXPORT FINAL LONG DATASET
# Save final cleaned analytical dataset for SQL and Power BI.

df_long.to_csv("inventory_2023_long.csv", index=False, encoding="utf-8-sig")
print("\nFile saved: inventory_2023_long.csv")